# 01 · Ingest — building the filing panel

Two questions decide everything downstream: **which firms**, and **which
filings**.

The panel is 10-K filings, 2019–2024, for firms that
1. have a **traded ticker** — the deterioration label needs an equity price,
   so a firm without one cannot be labelled and must not enter; and
2. filed a 10-K in **every** year — the headline text feature compares a
   firm's risk factors to *its own* prior year, so a filing gap produces a
   discontinuity rather than an observation.

EDGAR publishes a quarterly `form.idx` listing every filing made that
quarter, so 24 requests identify every 10-K filer in the window instead of
10,000 per-company lookups. Only after the filters do we hit the submissions
API once per surviving firm.

**This notebook is idempotent.** Everything already on disk is skipped, so
re-running it is cheap and safe.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import config
print(config.describe())

repo   : X:\Projects\edgar-credit-signal
data   : X:\Projects\edgar-credit-signal\data
filings: X:\Projects\edgar-credit-signal\data\raw\filings


## Where am I?

Run this first. It reports what is already built and what is next.

In [2]:
import status
status.main()


project data: X:\Projects\edgar-credit-signal\data

stage                             state                             
--------------------------------------------------------------------
1a build_universe                 8,878 filings, 1,499 firms
1b download_filings               8,875 files, 1,015 MB  (3 still missing)
2a extract_sections               8,875 rows, 7,907 usable Item 1A (89%)
2b text_features                  8,875 rows, 6,330 year-over-year observations

next:  python download_filings.py



## 1a · Build the universe

~5 minutes. Fetches 24 quarterly indexes (cached to disk), applies the two
filters, caps the panel, then resolves the primary document filename for
each filing.

In [6]:
import build_universe
build_universe.main()

2026-08-22 01:42:33,039 INFO fetching 24 quarterly form indexes
2026-08-22 01:42:40,349 INFO found 42,909 10-K filings by 10,099 distinct CIKs
2026-08-22 01:42:40,433 INFO after ticker filter: 23,873 filings, 4,606 firms
2026-08-22 01:42:40,451 INFO after balanced-panel filter: 3,190 firms
2026-08-22 01:42:40,464 INFO universe fixed at 1,500 firms
2026-08-22 01:42:40,465 INFO resolving primary documents (1500 requests)
submissions: 100%|██████████| 1500/1500 [03:07<00:00,  7.99it/s]
2026-08-22 01:45:48,594 INFO manifest written: 8,700 filings, 1,492 firms -> X:\Projects\edgar-credit-signal\data\raw\manifest.parquet


filing_year
2019    1360
2020    1419
2021    1461
2022    1481
2023    1487
2024    1492


##Download the filings


In [4]:
import download_filings
download_filings.main()

2026-08-22 00:43:24,615 INFO 8,700 filings in manifest
filings:  30%|██▉       | 2571/8700 [18:12<49:37,  2.06it/s]  2026-08-22 01:01:38,599 WARNING network error https://www.sec.gov/Archives/edgar/data/319201/000031920123000031/klac-20230630.htm (attempt 1): ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
2026-08-22 01:01:38,839 WARNING network error https://www.sec.gov/Archives/edgar/data/319201/000031920122000023/klac-20220630.htm (attempt 1): ('Connection broken: IncompleteRead(15731 bytes read, 365497 more expected)', IncompleteRead(15731 bytes read, 365497 more expected))
filings:  70%|███████   | 6091/8700 [41:12<12:35,  3.45it/s]2026-08-22 01:24:37,485 WARNING network error https://www.sec.gov/Archives/edgar/data/910073/000156459021009479/nycb-10k_20201231.htm (attempt 1): ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
2026-08-22 01:24:37,621 WA

## Panel summary

These are the numbers that go in the writeup, so read them off the data
rather than from memory.

In [5]:
man = pd.read_parquet(config.MANIFEST)
files = list(config.FILINGS.glob("*.txt.gz"))
mb = sum(p.stat().st_size for p in files) / 1e6

print(f"firms in panel        {man['cik'].nunique():>8,}")
print(f"filings in manifest   {len(man):>8,}")
print(f"downloaded            {len(files):>8,}   ({len(files)/len(man):.1%})")
print(f"on disk, gzipped      {mb:>8,.0f} MB")
print()
print(man.groupby("filing_year").size().rename("filings").to_frame().T.to_string())

firms in panel           1,492
filings in manifest      8,700
downloaded              17,419   (200.2%)
on disk, gzipped         1,932 MB

filing_year  2019  2020  2021  2022  2023  2024
filings      1360  1419  1461  1481  1487  1492
